## STAC/ZARR 2025 workshop

This notebook shows the following features:
- CADIP STAC search
- AUXIP STAC search
- Staging of S3A CADIP session as a STAC item
- Staging of S3A auxiliary files (AX___OSF_AX / AX___FRO_AX / AX___FPO_AX) as STAC items

## 1. Initialization

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *

init_demo()

# Reload the global vars again
from resources.utils import *

## 1. Find latest S3A CADIP session as a STAC item

In [ ]:
# Retrieve https://stac-browser-cadip.ops.rs-python.eu/collections/s3_sgs/items/S3A_20250612034536048530
# Here we show support of cql2-text filters in GET /search
cadip_session = cadip_client.search(method="GET", limit=1, max_items=1, collections="s3_sgs",
                                    stac_filter="platform=sentinel-3a and cadip:delivery_push_ok=true",
                                    sortby=[ { "field": "datetime", "direction": "desc" } ])[0]
cadip_session_href = cadip_session.get_links("self")[0].get_href()
print(f"CADIP session: {cadip_session_href}")

## 2. Find auxiliary data from AUXIP as STAC items

In [ ]:
# Retrieve https://rspy.ops.rs-python.eu/auxip/collections/S3-AX___OSF_AX/items/S3A_AX___OSF_AX_20160216T192404_99991231T235959_20240624T085540___________________EUM_O_AL_001.SEN3
# Here we show support of cql2-json filters in POST /search
osf = auxip_client.search(method="POST", limit=1, max_items=1, collections="S3-AX___OSF_AX", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "AX___OSF_AX" ] },
            { "op": "=", "args": [ { "property": "published" }, "2024-06-24T09:02:46.385000Z" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
osf_href = osf.get_links("self")[0].get_href()
print(f"OSF: {osf_href}")

# Retrieve https://stac-browser-auxip.ops.rs-python.eu/collections/S3-AX___FRO_AX/items/S3A_AX___FRO_AX_20250609T000000_20250619T000000_20250612T064635___________________EUM_O_AL_001.SEN3
fro = auxip_client.search(method="POST", limit=1, max_items=1, collections="S3-AX___FRO_AX", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "AX___FRO_AX" ] },
            { "op": "=", "args": [ { "property": "published" }, "2025-06-12T06:49:30.263000Z" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
fro_href = fro.get_links("self")[0].get_href()
print(f"FRO: {fro_href}")

# Retrieve https://stac-browser-auxip.ops.rs-python.eu/collections/S3-AX___FPO_AX/items/S3A_AX___FPO_AX_20250612T000000_20250619T000000_20250612T064819___________________EUM_O_AL_001.SEN3
fpo = auxip_client.search(method="POST", limit=1, max_items=1, collections="S3-AX___FPO_AX", stac_filter={
        "op": "and",
        "args": [
            { "op": "=", "args": [ { "property": "product:type" }, "AX___FPO_AX" ] },
            { "op": "=", "args": [ { "property": "published" }, "2025-06-12T06:54:39.878000Z" ] }
        ]
    }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
fpo_href = fpo.get_links("self")[0].get_href()
print(f"FPO: {fpo_href}")

## 3. Create Catalog collections

In [ ]:
temporal = TemporalExtent([datetime(2016, 2, 16), datetime.now()])

s3_sessions_coll = get_or_create_test_collection(
    collection_id="s3_sessions", description="S3 staged CADIP sessions", title="S3 sessions", temporal=temporal)
osf_coll = get_or_create_test_collection(
    collection_id="s3_aux_osf", description="S3-AX___OSF_AX staged auxiliary files", title="AX___OSF_AX", temporal=temporal)
fro_coll = get_or_create_test_collection(
    collection_id="s3_aux_fro", description="S3-AX___FRO_AX staged auxiliary files", title="AX___FRO_AX", temporal=temporal)
fpo_coll = get_or_create_test_collection(
    collection_id="s3_aux_fpo", description="S3-AX___FPO_AX staged auxiliary files", title="AX___FPO_AX", temporal=temporal)

print(f"S3 session collection: {s3_sessions_coll.get_links('self')[0].get_href()}")
print(f"OSF collection: {osf_coll.get_links('self')[0].get_href()}")
print(f"FRO collection: {fro_coll.get_links('self')[0].get_href()}")
print(f"FPO collection: {fpo_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

## 4. Stage session and auxiliary data in the STAC Catalog

In [ ]:
stage_single_item(osf, osf_coll)
stage_single_item(fro, fro_coll)
stage_single_item(fpo, fpo_coll)
stage_single_item(cadip_session, s3_sessions_coll)